# Relaciones de equivalencia

**Matemática Discreta (2026/27)** · Edgar Martínez-Moro · Departamento de Matemática Aplicada, Escuela de Ingeniería Informática, Universidad de Valladolid

Este cuaderno acompaña a la sesión 12 del tema de **relaciones binarias**. Contiene:

1. Cómo comprobar si una relación es de equivalencia y, si no lo es, por qué.
2. Clases de equivalencia, conjunto cociente y la matriz por bloques.
3. La biyección entre relaciones de equivalencia y particiones.
4. El núcleo de una aplicación (anagramas y congruencias).
5. Todas las particiones de un conjunto, números de Stirling y de Bell.
6. La estructura de unión-búsqueda (*union-find*).

## Repaso

Una relación $\mathcal R$ en $A$ es **de equivalencia** si es reflexiva, simétrica y transitiva. La **clase** de $x$ es $[x]=\{y\in A:(y,x)\in\mathcal R\}$ y el **conjunto cociente** es $A/\mathcal R=\{[x]:x\in A\}$. Las clases forman una partición de $A$ y, recíprocamente, cada partición define una relación de equivalencia: «estar en el mismo bloque». Así, $\mathcal R\mapsto A/\mathcal R$ es una biyección entre las relaciones de equivalencia $\mathcal E(A)$ y las particiones $\Pi(A)$.

En todo el cuaderno una relación es un **conjunto de pares** $(a,b)$.

In [1]:
from functools import lru_cache
from itertools import product
from math import comb
import time


def fmt(conjunto):
    """{1, 2, 3}"""
    return "{" + ", ".join(str(x) for x in conjunto) + "}"


def fmt_particion(P):
    """{{1, 3}, {2}}"""
    return "{" + ", ".join(fmt(B) for B in P) + "}"


def mostrar_matriz(elementos, R, titulo=None):
    """Matriz booleana de R; los ceros se escriben como puntos."""
    elementos = list(elementos)
    ancho = max(len(str(x)) for x in elementos)
    if titulo:
        print(titulo)
    print(" " * (ancho + 1) + " ".join(str(b).rjust(ancho) for b in elementos))
    for a in elementos:
        fila = ("1" if (a, b) in R else "." for b in elementos)
        print(str(a).rjust(ancho) + " " + " ".join(c.rjust(ancho) for c in fila))

## 1. ¿Es de equivalencia?

`fallo_equivalencia` devuelve `None` si la relación es de equivalencia y, si no, un texto con la primera propiedad que falla y un contraejemplo. `product(A, repeat=2)` recorre $A\times A$.

In [2]:
def fallo_equivalencia(elementos, R):
    """None si R es de equivalencia en `elementos`; si no, el primer fallo."""
    for x in elementos:
        if (x, x) not in R:
            return f"no es reflexiva: falta ({x}, {x})"
    for x, y in product(elementos, repeat=2):
        if (x, y) in R and (y, x) not in R:
            return f"no es simétrica: está ({x}, {y}), falta ({y}, {x})"
    for x, y, z in product(elementos, repeat=3):
        if (x, y) in R and (y, z) in R and (x, z) not in R:
            return (f"no es transitiva: están ({x}, {y}) y ({y}, {z}),"
                    f" falta ({x}, {z})")
    return None


def es_equivalencia(elementos, R):
    return fallo_equivalencia(elementos, R) is None


def congruencia(m, elementos):
    """a R b  si a - b es múltiplo de m."""
    return {(a, b) for a in elementos for b in elementos if (a - b) % m == 0}

In [3]:
A = [1, 2, 3, 4, 5, 6]
print(fallo_equivalencia(A, congruencia(3, A)))

None


La congruencia módulo $3$ es de equivalencia. En particular $(3,3)$ está, porque $3-3=0$ es múltiplo de $3$.

Las tres relaciones siguientes no son congruencias: son conjuntos de pares escritos a mano, y cada una falla en una sola propiedad.

In [4]:
R2 = {(1, 1), (2, 2), (1, 2), (2, 1)}               # en {1, 2, 3}
print(fallo_equivalencia([1, 2, 3], R2))

no es reflexiva: falta (3, 3)


In [5]:
R3 = {(1, 1), (2, 2), (1, 2)}                       # en {1, 2}
print(fallo_equivalencia([1, 2], R3))

no es simétrica: está (1, 2), falta (2, 1)


In [6]:
R4 = {(1, 1), (2, 2), (3, 3), (1, 2), (2, 1), (2, 3), (3, 2)}
print(fallo_equivalencia([1, 2, 3], R4))

no es transitiva: están (1, 2) y (2, 3), falta (1, 3)


`R2` es simétrica y transitiva, pero no reflexiva: el $3$ no aparece en ningún par, ni siquiera en $(3,3)$. Por eso el «argumento» de que la reflexividad se deduce de las otras dos ($x\,\mathcal R\,y\Rightarrow y\,\mathcal R\,x\Rightarrow x\,\mathcal R\,x$) es falso: necesita que $x$ esté relacionado con algún $y$.

## 2. Clases de equivalencia y conjunto cociente

In [7]:
def clase(x, elementos, R):
    """[x] = {y : (y, x) en R}"""
    return [y for y in elementos if (y, x) in R]


def conjunto_cociente(elementos, R):
    """Lista de clases de equivalencia (cada una, una lista)."""
    cociente, vistos = [], set()
    for x in elementos:
        if x not in vistos:              # x aún no tiene clase
            c = clase(x, elementos, R)
            cociente.append(c)
            vistos.update(c)
    return cociente


def relacion_de_particion(particion):
    """x R y  si x e y están en el mismo bloque."""
    return {(x, y) for B in particion for x in B for y in B}


def ordenar_por_clases(elementos, R):
    """Los elementos reordenados clase a clase: la matriz queda por bloques."""
    return [x for c in conjunto_cociente(elementos, R) for x in c]

In [8]:
R = congruencia(3, A)
print("[1] =", clase(1, A, R))
print("A/R =", conjunto_cociente(A, R))

[1] = [1, 4]
A/R = [[1, 4], [2, 5], [3, 6]]


Si se ordenan los elementos clase a clase, la matriz de una relación de equivalencia queda formada por **bloques de unos** en la diagonal, uno por clase (los ceros se escriben como puntos):

In [9]:
mostrar_matriz(A, R, "En el orden 1, ..., 6:")
print()
mostrar_matriz(ordenar_por_clases(A, R), R, "Clase a clase:")

En el orden 1, ..., 6:
  1 2 3 4 5 6
1 1 . . 1 . .
2 . 1 . . 1 .
3 . . 1 . . 1
4 1 . . 1 . .
5 . 1 . . 1 .
6 . . 1 . . 1

Clase a clase:
  1 4 2 5 3 6
1 1 1 . . . .
4 1 1 . . . .
2 . . 1 1 . .
5 . . 1 1 . .
3 . . . . 1 1
6 . . . . 1 1


In [10]:
B = list(range(12))
R5 = congruencia(5, B)
print(conjunto_cociente(B, R5))
mostrar_matriz(ordenar_por_clases(B, R5), R5)

[[0, 5, 10], [1, 6, 11], [2, 7], [3, 8], [4, 9]]
    0  5 10  1  6 11  2  7  3  8  4  9
 0  1  1  1  .  .  .  .  .  .  .  .  .
 5  1  1  1  .  .  .  .  .  .  .  .  .
10  1  1  1  .  .  .  .  .  .  .  .  .
 1  .  .  .  1  1  1  .  .  .  .  .  .
 6  .  .  .  1  1  1  .  .  .  .  .  .
11  .  .  .  1  1  1  .  .  .  .  .  .
 2  .  .  .  .  .  .  1  1  .  .  .  .
 7  .  .  .  .  .  .  1  1  .  .  .  .
 3  .  .  .  .  .  .  .  .  1  1  .  .
 8  .  .  .  .  .  .  .  .  1  1  .  .
 4  .  .  .  .  .  .  .  .  .  .  1  1
 9  .  .  .  .  .  .  .  .  .  .  1  1


## 3. La biyección entre $\mathcal E(A)$ y $\Pi(A)$

`conjunto_cociente` va de la relación a la partición y `relacion_de_particion` hace el camino de vuelta. Comprobamos que una es la inversa de la otra.

In [11]:
P = [[1, 3], [2], [4, 5, 6]]
Q = relacion_de_particion(P)
print(es_equivalencia(A, Q), conjunto_cociente(A, Q))
mostrar_matriz(A, Q)

True [[1, 3], [2], [4, 5, 6]]
  1 2 3 4 5 6
1 1 . 1 . . .
2 . 1 . . . .
3 1 . 1 . . .
4 . . . 1 1 1
5 . . . 1 1 1
6 . . . 1 1 1


## 4. El núcleo de una aplicación

Si $f:A\to B$ es una aplicación, $a\,\mathcal R\,b\iff f(a)=f(b)$ es una relación de equivalencia (el **núcleo** de $f$). Sus clases son los conjuntos de elementos con la misma imagen. De hecho, toda relación de equivalencia es el núcleo de la aplicación $x\mapsto[x]$.

Con $f(w)=$ las letras de $w$ ordenadas, dos palabras están relacionadas si son **anagramas**:

In [12]:
def nucleo(f, elementos):
    """a R b  si f(a) == f(b)  (núcleo de la aplicación f)."""
    return {(a, b) for a in elementos for b in elementos if f(a) == f(b)}

In [13]:
pal = ["amor", "saco", "roma", "casa", "mora",
       "cosa", "ramo", "caso", "asco", "saca"]
letras = lambda w: "".join(sorted(w))      # "roma" -> "amor"
for c in conjunto_cociente(pal, nucleo(letras, pal)):
    print(c)

['amor', 'roma', 'mora', 'ramo']
['saco', 'cosa', 'caso', 'asco']
['casa', 'saca']


La congruencia módulo $m$ es el núcleo de $f(a)=$ resto de dividir $a$ entre $m$:

In [14]:
B = range(-10, 20)
print(congruencia(5, B) == nucleo(lambda a: a % 5, B))

True


## 5. Todas las particiones. Números de Stirling y de Bell

Para repartir $\{a_1,\dots,a_n\}$ se reparte primero $\{a_2,\dots,a_n\}$ y después $a_1$ va a un bloque nuevo o a uno de los bloques que ya hay. Las mismas dos opciones dan las recurrencias

$$S(n,k)=S(n-1,k-1)+k\,S(n-1,k),\qquad B(n+1)=\sum_{k=0}^{n}\binom nk B(k).$$

In [15]:
def particiones(elementos):
    """Genera todas las particiones de `elementos` (listas de bloques)."""
    elementos = list(elementos)
    if not elementos:
        yield []
        return
    primero, resto = elementos[0], elementos[1:]
    for P in particiones(resto):
        yield [[primero]] + P                  # en un bloque nuevo
        for i in range(len(P)):                # o en un bloque de P
            yield P[:i] + [[primero] + P[i]] + P[i + 1:]


@lru_cache(maxsize=None)
def stirling2(n, k):
    """S(n, k) = S(n-1, k-1) + k S(n-1, k)"""
    if n == k:
        return 1
    if k == 0 or k > n:
        return 0
    return stirling2(n - 1, k - 1) + k * stirling2(n - 1, k)


def bell(n):
    """B(n) con la recurrencia B(m+1) = sum_k C(m, k) B(k)."""
    B = [1]
    for m in range(n):
        B.append(sum(comb(m, k) * B[k] for k in range(m + 1)))
    return B[n]

In [16]:
for P in particiones([1, 2, 3]):
    print(fmt_particion(P))

{{1}, {2}, {3}}
{{1, 2}, {3}}
{{2}, {1, 3}}
{{1}, {2, 3}}
{{1, 2, 3}}


In [17]:
print("Números de Bell:", [bell(n) for n in range(1, 13)])
print()
print("Números de Stirling de segunda especie S(n, k):")
for n in range(1, 9):
    fila = [stirling2(n, k) for k in range(1, n + 1)]
    print(f"n = {n}:", " ".join(f"{x:>5}" for x in fila), f"   suma = {sum(fila)}")

Números de Bell: [1, 2, 5, 15, 52, 203, 877, 4140, 21147, 115975, 678570, 4213597]

Números de Stirling de segunda especie S(n, k):
n = 1:     1    suma = 1
n = 2:     1     1    suma = 2
n = 3:     1     3     1    suma = 5
n = 4:     1     7     6     1    suma = 15
n = 5:     1    15    25    10     1    suma = 52
n = 6:     1    31    90    65    15     1    suma = 203
n = 7:     1    63   301   350   140    21     1    suma = 877
n = 8:     1   127   966  1701  1050   266    28     1    suma = 4140


In [18]:
# El generador produce exactamente B(n) particiones, sin repetir ninguna
for n in range(1, 9):
    L = [tuple(sorted(tuple(sorted(B)) for B in P)) for P in particiones(range(n))]
    assert len(L) == len(set(L)) == bell(n)
print("comprobado hasta n = 8")

comprobado hasta n = 8


### Comprobación por fuerza bruta

Cada relación en un conjunto de $n$ elementos es una elección de sí o no para cada uno de los $n^2$ pares: hay $2^{n^2}$. ¿Cuántas son de equivalencia?

In [19]:
def todas_las_relaciones(elementos):
    """Genera las 2^(n^2) relaciones en `elementos`."""
    pares = [(a, b) for a in elementos for b in elementos]
    for bits in product((0, 1), repeat=len(pares)):
        yield {p for p, b in zip(pares, bits) if b}

In [20]:
for n in range(1, 5):
    E = list(range(1, n + 1))
    t0 = time.perf_counter()
    eqv = sum(es_equivalencia(E, R) for R in todas_las_relaciones(E))
    print(f"n = {n}: {2 ** (n * n):>6} relaciones, {eqv:>2} de equivalencia"
          f"  (B({n}) = {bell(n)}, {time.perf_counter() - t0:.2f} s)")

n = 1:      2 relaciones,  1 de equivalencia  (B(1) = 1, 0.00 s)
n = 2:     16 relaciones,  2 de equivalencia  (B(2) = 2, 0.00 s)
n = 3:    512 relaciones,  5 de equivalencia  (B(3) = 5, 0.00 s)


n = 4:  65536 relaciones, 15 de equivalencia  (B(4) = 15, 0.10 s)


Con $n=5$ habría que revisar $2^{25}\approx 3{,}4\cdot10^7$ relaciones: la fórmula es mucho mejor que contar.

## 6. Unión-búsqueda (*union-find*)

A veces sólo se conocen **algunos** pares de una equivalencia y los demás se deducen por simetría y transitividad. La estructura de unión-búsqueda (Galler y Fischer, 1964) calcula las clases sin escribir la relación entera:

- Cada clase es un **árbol**: cada elemento apunta a su padre y la **raíz** representa a la clase. Al principio cada elemento es su propia raíz.
- **Buscar** $x$: subir de padre en padre hasta la raíz.
- **Unir** $a$ y $b$: si sus raíces son distintas, colgar una de la otra.

La línea `padre[x] = padre[padre[x]]` acorta los caminos mientras se busca. Si además se cuelga siempre el árbol pequeño del grande, cada operación cuesta en la práctica una constante (Tarjan, 1975).

Lo que se obtiene son las clases de la **menor** relación de equivalencia que contiene a los pares dados. En la sesión 13 veremos que esa relación es $tsr(\mathcal R)$.

In [21]:
def clases_union_find(elementos, pares):
    """
    Clases de la menor equivalencia que contiene a los pares dados.
    Cada clase es un árbol; su raíz es el representante.
    """
    padre = {x: x for x in elementos}      # todos empiezan solos

    def raiz(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]     # acorta el camino
            x = padre[x]
        return x

    for a, b in pares:
        ra, rb = raiz(a), raiz(b)
        if ra != rb:
            padre[rb] = ra                 # une las dos clases
    clases = {}
    for x in elementos:
        clases.setdefault(raiz(x), []).append(x)
    return list(clases.values())


def mostrar_union_find(elementos, pares):
    """Igual que clases_union_find, pero imprime cada paso."""
    elementos = list(elementos)
    padre = {x: x for x in elementos}

    def raiz(x):
        while padre[x] != x:
            x = padre[x]
        return x

    def clases():
        c = {}
        for x in elementos:
            c.setdefault(raiz(x), []).append(x)
        return list(c.values())

    filas = []
    for a, b in pares:
        ra, rb = raiz(a), raiz(b)
        if ra != rb:
            padre[rb] = ra
            accion = f"padre[{rb}] = {ra}"
        else:
            accion = "ya juntos"
        filas.append((f"({a}, {b})", f"{ra}, {rb}", accion,
                      " ".join(fmt(c) for c in clases())))
    cab = ("par", "raíces", "acción", "clases")
    anchos = [max(len(f[i]) for f in filas + [cab]) for i in range(3)]
    for f in [cab] + filas:
        print("  " + "  ".join(c.ljust(w) for c, w in zip(f, anchos)) + "  " + f[3])
    print("  Representantes:", ", ".join(f"{raiz(c[0])} -> {fmt(c)}" for c in clases()))
    return clases()

In [22]:
pares = [(1, 5), (2, 6), (5, 8), (3, 7), (8, 1), (6, 7)]
mostrar_union_find(range(1, 9), pares);

  par     raíces  acción        clases
  (1, 5)  1, 5    padre[5] = 1  {1, 5} {2} {3} {4} {6} {7} {8}
  (2, 6)  2, 6    padre[6] = 2  {1, 5} {2, 6} {3} {4} {7} {8}
  (5, 8)  1, 8    padre[8] = 1  {1, 5, 8} {2, 6} {3} {4} {7}
  (3, 7)  3, 7    padre[7] = 3  {1, 5, 8} {2, 6} {3, 7} {4}
  (8, 1)  1, 1    ya juntos     {1, 5, 8} {2, 6} {3, 7} {4}
  (6, 7)  2, 3    padre[3] = 2  {1, 5, 8} {2, 3, 6, 7} {4}
  Representantes: 1 -> {1, 5, 8}, 2 -> {2, 3, 6, 7}, 4 -> {4}


El par $(8,1)$ no cambia nada: $8$ y $1$ ya tenían la misma raíz. Con $(6,7)$ se juntan las clases de raíces $2$ y $3$.

### Comparación de tiempos

Clases de la congruencia módulo $7$ en $\{0,\dots,N-1\}$. Con `conjunto_cociente` hay que construir la relación entera (unos $N^2/7$ pares); a la unión-búsqueda le bastan los $N-7$ pares $(a,a+7)$.

In [23]:
for N in (500, 1000, 2000, 4000):
    E = list(range(N))
    t0 = time.perf_counter()
    R = congruencia(7, E)
    c1 = conjunto_cociente(E, R)
    t1 = time.perf_counter()
    c2 = clases_union_find(E, [(a, a + 7) for a in range(N - 7)])
    t2 = time.perf_counter()
    assert sorted(map(sorted, c1)) == sorted(map(sorted, c2))
    print(f"N = {N:>4}: {len(R):>8} pares | cociente {t1 - t0:7.3f} s"
          f" | unión-búsqueda {t2 - t1:.5f} s")

N =  500:    35716 pares | cociente   0.020 s | unión-búsqueda 0.00035 s


N = 1000:   142858 pares | cociente   0.102 s | unión-búsqueda 0.00066 s


N = 2000:   571430 pares | cociente   0.317 s | unión-búsqueda 0.00129 s


N = 4000:  2285716 pares | cociente   1.383 s | unión-búsqueda 0.00326 s


## Referencias

- Galler, B. A. y Fischer, M. J. (1964). An improved equivalence algorithm. *Communications of the ACM*, 7(5), 301–303.
- Tarjan, R. E. (1975). Efficiency of a good but not linear set union algorithm. *Journal of the ACM*, 22(2), 215–225.
- Grimaldi, R. P. (1998). *Matemática discreta y combinatoria* (3ª ed.). Addison Wesley Longman.
- Rosen, K. H. (2004). *Matemática discreta y sus aplicaciones* (5ª ed.). McGraw-Hill.